# Mini Project 1 — ChatGPT mobile reviews (public Play Store archive)

**Author:** Zhian Hu  
**Course:** HCDE 530 — Data Science for UX / HCI  
**Date:** May 2026  

Standalone analytical narrative: **what we examined**, **what we learned**, **what it implies** for HCI folks studying perception of ChatGPT‑class assistants.  

Bundled **`data/chatgpt_reviews_sample.csv`** is a reproducible **`n = 70,000` random sample** (~9 MiB so GitHub accepts the repo); it **preserves the population’s star-rating imbalance** (~three‑quarters five-star in the storefront feed). Full files: **[Kaggle: ChatGPT reviews (daily updated)](https://www.kaggle.com/datasets/ashishkumarak/chatgpt-reviews-daily-updated)** (`ashishkumarak`). Operational detail: **`README.md`** in this folder.


In [1]:
# Required for grading — do not delete the next line.
!pip install jupyter plotly kaleido pandas


zsh:1: command not found: pip


In [2]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

for _pkg in ("nbformat>=4.2.0",):
    try:
        import nbformat  # noqa: F401
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", _pkg, "-q"]
        )

MP1_DIR = Path.cwd().resolve()
DATA_PATH = MP1_DIR / "data" / "chatgpt_reviews_sample.csv"
FIG_DIR = MP1_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), (
    f"Missing {DATA_PATH}. Open/run from MP1 folder (see README.md)."
)
print("Notebook root:", MP1_DIR)
print("Data:", DATA_PATH.resolve())


Notebook root: /Users/hza25/Desktop/hcde530/MP1
Data: /Users/hza25/Desktop/hcde530/MP1/data/chatgpt_reviews_sample.csv


---

## Section 1 — Overview

### Dataset (for newcomers)
Rows are **individual Google Play reviews** scraped into an open **[Kaggle dataset titled “ChatGPT reviews [DAILY UPDATED]”](https://www.kaggle.com/datasets/ashishkumarak/chatgpt-reviews-daily-updated)**. Fields include textual **`content`**, integer **`score` (stars)**, timestamp **`at`**, hashed identifiers, optional **`reviewCreatedVersion` / `appVersion`**, and **`thumbsUpCount`**—“was this helpful?” tallies surfaced in the storefront UI.

### Questions & stakes
1. **Polarity:** How overwhelmingly **positive** is the corpus (dominant 5★)?  
2. **Engagement vs. polarity:** Do **average thumbs-ups** escalate for low-star critiques (potential “pile-on visibility”) versus praise?  
3. **Completeness:** Do **holes in version strings** sabotage rollout studies?

### Why HCI / UX?
Interfaces that host model updates need **signals of trust damage** buried under megabytes of fluff. Structured numeric evidence—before expensive qualitative reads—guides **risk triage**.


---

## Section 2 — Data Profile


In [3]:
df = pd.read_csv(DATA_PATH)
rating_col = "score"
print(df.shape)


(70000, 8)


Run **`df.head()`**

In [4]:
df.head()

,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion
0,f40be9f9-5092-485b-a717-796d681b4f43,Shaik Taju,EDITOR KA BAPP 〽️❌⚠️‼️,5,0,1.2025.168,2025-06-29 10:23:48,1.2025.168
1,79508b76-6b62-4adc-8368-5a4d14602bee,Sanjay Mandal,Sanjay Mandal like 👍,5,0,1.2025.350,2026-01-02 01:32:05,1.2025.350
2,a55d0876-6892-47df-ab22-9bb962833405,Binod kumar Shaw,verry good 👍,5,0,1.2024.352,2024-12-27 14:18:25,1.2024.352
3,f08b33f6-950a-4529-92f0-b65cc2cc2d09,jaydenwen ting,This will get you through school,5,0,1.2025.154,2025-06-12 08:49:38,1.2025.154
4,857aec71-2965-42fd-8e40-2b0533ac76db,Hamza Haouache,Great app ❤,5,0,1.2024.115,2024-05-02 12:12:43,1.2024.115


**Interpretation:** Early rows juxtapose terse emoji posts with substantive complaints—a reminder that polarity lives alongside **thin or noisy textual signals** needing cleaning before NLP.

Run **`df.info()`**

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              70000 non-null  object
 1   userName              70000 non-null  object
 2   content               69999 non-null  object
 3   score                 70000 non-null  int64 
 4   thumbsUpCount         70000 non-null  int64 
 5   reviewCreatedVersion  64903 non-null  object
 6   at                    70000 non-null  object
 7   appVersion            64903 non-null  object
dtypes: int64(2), object(6)
memory usage: 4.3+ MB


**Interpretation:** Seventy thousand reviews across eight labeled columns confirms low memory footprint versus the million-row parent file while showing **paired null drops** lining up on dual version strings.

Run **`df.describe()`**

In [6]:
df.describe()

,score,thumbsUpCount
count,70000.000000,70000.000000
mean,4.495971,0.126614
std,1.108990,6.625952
min,1.000000,0.000000
25%,5.000000,0.000000
50%,5.000000,0.000000
75%,5.000000,0.000000
max,5.000000,941.000000


**Interpretation:** Stars remain **bounded ordinals**, while thumbs-ups cluster near zero yet sport long tails—consistent with storefront feedback where handfuls of engagements dominate.

Run **`df.isnull().sum()`**

In [7]:
df.isnull().sum()

reviewId                   0
userName                   0
content                    1
score                      0
thumbsUpCount              0
reviewCreatedVersion    5097
at                         0
appVersion              5097
dtype: int64

**Interpretation:** Missingness stacks on **`reviewCreatedVersion` / `appVersion`** jointly, implying operational blind spots—not random typos—for anything mapping complaints to binaries.

---

## Section 3 — Analysis

Three visuals—**(1) donut composition, (2) line + markers trend, (3) treemap of missing totals**—map to the MP1 questions above. Kaleido persists each PNG to **`figures/`**; Markdown embeds previews for reviewers without live Plotly kernels.


### Chart 1 — Star composition (skew / polarity dominance)

**Answers:** Question 1.

![](./figures/chart1_rating_distribution.png)


In [8]:
counts = df[rating_col].value_counts().sort_index()
pie_labels = [f"{star}★" for star in counts.index]

fig1 = go.Figure(
    data=[
        go.Pie(
            labels=pie_labels,
            values=counts.values,
            hole=0.45,
            textinfo="label+percent",
            textposition="outside",
            sort=False,
        )
    ]
)
fig1.update_layout(
    title={
        "text": "Composition of ChatGPT Play Store ratings (share by star level, random sample)<br><sub>Wedges show review share; imbalance reflects Play Store—not equal stratification.</sub>",
        "x": 0.5,
        "xanchor": "center",
    },
    showlegend=False,
    margin=dict(t=130, l=40, r=40, b=80),
)

fig1.write_image(FIG_DIR / "chart1_rating_distribution.png")
fig1.show()


**Reading:** Majority share belongs to five-star applause—dissatisfied slices are comparatively **narrow wedges**, so exploratory qualitative passes must purposely **oversample** rare stars or risk caricaturing “typical” users.

### Chart 2 — Mean thumbs-up curve by star rating

**Answers:** Question 2.

![](./figures/chart2_mean_thumbs_by_rating.png)


In [9]:
means = (
    df.groupby(rating_col, as_index=False)["thumbsUpCount"]
    .mean()
    .sort_values(rating_col)
)

fig2 = go.Figure(
    data=[
        go.Scatter(
            x=means[rating_col],
            y=means["thumbsUpCount"],
            mode="lines+markers",
            line=dict(width=3, shape="linear"),
            marker=dict(size=12),
        )
    ]
)
fig2.update_layout(
    title="Average reader thumbs-ups by star rating (trend across ordered scale)",
    xaxis_title="Star rating (1–5 ordinal scale)",
    yaxis_title="Mean thumbs-up count (per review)",
    xaxis=dict(tickmode="linear", dtick=1),
)

fig2.write_image(FIG_DIR / "chart2_mean_thumbs_by_rating.png")
fig2.show()


**Reading:** Connecting ordinal stars highlights a **muted slope**—thumbs averages do **not** explode alongside critique stars here, so grievance amplification is subtler than a surface aggregate suggests.

### Chart 3 — Treemap of missing cells

**Answers:** Question 3.

![](./figures/chart3_missing_values_by_column.png)


In [10]:
missing_series = df.isnull().sum()
missing_series = missing_series[missing_series > 0].sort_values(ascending=False)
missing_tbl = pd.DataFrame(
    {
        "column": missing_series.index.astype(str),
        "missing_cells": missing_series.values,
    }
)

fig3 = px.treemap(
    missing_tbl,
    path=["column"],
    values="missing_cells",
    title="Missing cells by column (tile area proportional to missing count)",
)
fig3.update_traces(textinfo="label+value")
fig3.update_layout(margin=dict(t=48, l=12, r=12, b=12))

fig3.write_image(FIG_DIR / "chart3_missing_values_by_column.png")
fig3.show()


**Reading:** Two rectangles dominate (**version fields**) while core sentiment columns stay comparatively solid—a pattern demanding **explicit “unknown_version” tagging** whenever patch narratives matter.

---

## Section 4 — Conclusions

**Polarity skew —** storefront samples flood with glowing five-star chatter; dissatisfied voices exist but statistically thin, so humane sampling pipelines should **preferentially retain** scarce low-star strata before thematic coding conclusions.  

**Engagement linkage —** mean helpfulness votes plateau across stars—a **possibly null analytic insight** implying thumbs metrics cannot proxy complaint salience absent careful covariates (review length recency influencer effects).  

**Data hazards —** dual version identifiers share missingness footprints; attempting longitudinal blame without guarding those gaps biases inference. Immediate patch: annotate missingness tiers then compare text themes inside each tier.  

**Future digs —** model seasonal surge windows on **`at`**, correlate text length readability with thumbs spikes, enrich with multilingual sentiment baselines—all requiring compute beyond tonight’s exploratory slice.


---

## Section 5 — Process story *(self-reflection; optional spotlight)*

**Search & scaffolding:** Weeks 4–5 forced me onto a single open corpus with reproducible ingestion (Kaggle). **A5** pandas drills (`head/info/value_counts/groupby/isnull`) became the argumentative spine for **`df.describe`** arguments here.  

**Pivot / constraint engineering:** Instructor demands *standalone runnable artifacts*. Shipping the **130 MB** master CSV violates GitHub’s hard caps, so I re-sampled seventy thousand rows via **deterministic RNG** seeded for auditability—the proportionality story stays faithful while cloning-friendly.  

**AI surprise / tooling:** Debugging Plotly’s HTML export taught me **`nbformat`** as a lurking dependency whenever `show()` interacts with Jupyter mime bundles—small but irritating infrastructure lesson. GPT-style assistants accelerated syntax checks yet **numbers** stem from committed CSV + notebook code I re-ran.  

**If I lost work:** Seeds + scripted sampling mean regeneration is repeatable from the cached Kaggle download—process discipline mattered more than heroically hand-editing spreadsheets.
